(sec-9)=
# 9. Reproducibilidad y bibliografía

(sec-9-1)=
## 9.1 Entorno, semillas y ejecución

- **Entorno:** `conda create -n bm_e3 python=3.12`, `pip install -r requirements.txt` y `python -m ipykernel install --user --name bm_e3`. Semilla global `SEED = 42` en `src/bm/config.py`; el L2E es determinista y la semilla solo afecta a Optuna.
- **Datos:** `data/bank-additional-full.csv` (UCI, CC BY 4.0) y los tres conjuntos de OpenML (identificadores 1169, 151 y 1487), que `sklearn.datasets.fetch_openml` descarga en `data/adicionales/` la primera vez.
- **Orden de ejecución** (desde la raíz del proyecto): `python scripts/importar_e3.py` (resultados del tercer entregable), `python scripts/ejecutar_original.py` (validación anidada, ablación, semillas y modelos finales; unos 6 minutos con 10 núcleos), `python scripts/ejecutar_adicionales.py` (conjuntos adicionales; unos 25 minutos), `python scripts/ejecutar_analisis.py` (sensibilidad, robustez, complejidad y SHAP), `python scripts/revision_literatura.py` y `python scripts/verificar_referencias.py` (revisión y bibliografía), `pytest -q tests` y `python scripts/construir_cuaderno.py --ejecutar`. Para el dashboard y el libro (en el repositorio, desde `cuarto_entregable/`): `python scripts/exportar_dashboard_l2e.py --app ..` (artefactos y resultados del L2E en el dashboard), `python scripts/capturas_dashboard.py --url <URL del dashboard>` (capturas del capítulo 7) y `python scripts/construir_libro.py --libro ../book` (fuentes del Jupyter Book, que se compila con `jupyter-book build book/`).

In [32]:
display(Markdown("### 9.2 Tabla maestra de experimentos"))
m_e3 = maestra[["tarea", "modelo", "balanceo", "optimizador", "auc_pr_media", "auc_pr_de", "auc_media", "auc_de",
                "rmse_media", "rmse_de", "r2_media", "t_opt_total", "t_fit_medio", "t_pred_medio", "seed"]].copy()
m_e3["origen"] = "Tercer entregable"
filas = []
for v in VARIANTES:
    c_ = corr[v]
    filas.append({"tarea": "clf", "modelo": f"l2e:{v}", "balanceo": "ninguno", "optimizador": "bayes",
                  "auc_pr_media": np.mean([c["auc_pr"] for c in c_]), "auc_pr_de": np.std([c["auc_pr"] for c in c_], ddof=1),
                  "auc_media": np.mean([c["auc"] for c in c_]), "auc_de": np.std([c["auc"] for c in c_], ddof=1),
                  "t_opt_total": sum(c["t_opt"] for c in c_), "t_fit_medio": np.mean([c["t_fit"] for c in c_]),
                  "t_pred_medio": np.mean([c["t_pred"] for c in c_]), "seed": SEED, "origen": "Modelo original"})
for s, c_ in corr_sem.items():
    filas.append({"tarea": "clf", "modelo": "l2e:l2e", "balanceo": "ninguno", "optimizador": "bayes",
                  "auc_pr_media": np.mean([c["auc_pr"] for c in c_]), "auc_pr_de": np.std([c["auc_pr"] for c in c_], ddof=1),
                  "auc_media": np.mean([c["auc"] for c in c_]), "auc_de": np.std([c["auc"] for c in c_], ddof=1),
                  "t_opt_total": sum(c["t_opt"] for c in c_), "t_fit_medio": np.mean([c["t_fit"] for c in c_]),
                  "t_pred_medio": np.mean([c["t_pred"] for c in c_]), "seed": s, "origen": "Modelo original (semilla)"})
maestra_e4 = pd.concat([m_e3, pd.DataFrame(filas)], ignore_index=True)
maestra_e4["prueba_auc_pr"] = maestra_e4["modelo"].map(lambda m: fin_l2e[m.split(":")[1]]["auc_pr"] if m.startswith("l2e:") else np.nan)
maestra_e4.loc[maestra_e4["origen"] == "Modelo original (semilla)", "prueba_auc_pr"] = [fin_sem[s]["auc_pr"] for s in corr_sem]
maestra_e4.to_csv(RES / "tabla_maestra_e4.csv", index=False)
maestra_e4.to_parquet(RES / "tabla_maestra_e4.parquet", index=False)
tabla(maestra_e4[maestra_e4["tarea"] == "clf"].sort_values("auc_pr_media", ascending=False).head(15), dec=4, indice=False)
n_e3 = maestra_e4["origen"].eq("Tercer entregable")
print(f"Tabla maestra: {len(maestra_e4)} filas ({n_e3.sum()} combinaciones del tercer entregable, "
      f"{(n_e3 & maestra_e4['tarea'].eq('clf')).sum()} de clasificación y {(n_e3 & maestra_e4['tarea'].eq('reg')).sum()} de "
      f"regresión, y {(~n_e3).sum()} corridas del modelo original) en resultados/tabla_maestra_e4.csv y .parquet")
assert n_e3.sum() == 140

(sec-9-2)=
## 9.2 Tabla maestra de experimentos

tarea,modelo,balanceo,optimizador,auc_pr_media,auc_pr_de,auc_media,auc_de,rmse_media,rmse_de,r2_media,t_opt_total,t_fit_medio,t_pred_medio,seed,origen,prueba_auc_pr
clf,l2e:sin_pendientes,ninguno,bayes,"0,1152","0,0974","0,5818","0,0837",—,—,—,"188,5084","0,8676","0,2134",42,Modelo original,"0,5713"
clf,l2e:l2e,ninguno,bayes,"0,1107","0,1156","0,5598","0,1017",—,—,—,"195,4794","0,9101","0,2541",42,Modelo original,"0,6019"
clf,l2e:sin_nivel_latente,ninguno,bayes,"0,1099","0,1203","0,5532","0,1076",—,—,—,"193,3446","0,8529","0,2595",42,Modelo original,"0,5968"
clf,l2e:l2e,ninguno,bayes,"0,1091","0,1162","0,5568","0,1017",—,—,—,"191,4051","0,8851","0,2893",1,Modelo original (semilla),"0,6035"
clf,l2e:l2e,ninguno,bayes,"0,1083","0,1166","0,5565","0,1035",—,—,—,"185,6307","0,8457","0,2579",2,Modelo original (semilla),"0,5966"
clf,l2e:l2e,ninguno,bayes,"0,1068","0,1151","0,5538","0,1028",—,—,—,"188,9591","0,8921","0,2768",3,Modelo original (semilla),"0,5926"
clf,l2e:sin_incertidumbre,ninguno,bayes,"0,1060","0,1140","0,5597","0,1031",—,—,—,"191,5351","0,8510","0,2199",42,Modelo original,"0,5935"
clf,naive_bayes,class_weight,bayes,"0,1028","0,1081","0,5368","0,1019",—,—,—,"151,4339","0,6616","0,3423",42,Tercer entregable,—
clf,naive_bayes,smote,genetico,"0,0997","0,0955","0,5405","0,0771",—,—,—,"167,7956","0,7896","0,3051",42,Tercer entregable,—
clf,naive_bayes,smote,random,"0,0997","0,0958","0,5405","0,0762",—,—,—,"168,0762","0,8392","0,3538",42,Tercer entregable,—


Tabla maestra: 150 filas (140 combinaciones del tercer entregable, 112 de clasificación y 28 de regresión, y 10 corridas del modelo original) en resultados/tabla_maestra_e4.csv y .parquet


(sec-9-3)=
## 9.3 Referencias

Las entradas en formato BibTeX están en `resultados/literatura/references.bib`; las que tienen DOI se generaron desde doi.org con `scripts/verificar_referencias.py`.

- Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining*, 2623-2631. <https://doi.org/10.1145/3292500.3330701>
- Benavoli, A., Corani, G., Demšar, J. y Zaffalon, M. (2017). Time for a change: A tutorial for comparing multiple classifiers through Bayesian analysis. *Journal of Machine Learning Research*, 18(77), 1-36.
- Bergstra, J., Bardenet, R., Bengio, Y. y Kégl, B. (2011). Algorithms for hyper-parameter optimization. *Advances in Neural Information Processing Systems*, 24.
- Boyd, S. y Vandenberghe, L. (2004). *Convex optimization*. Cambridge University Press. <https://doi.org/10.1017/CBO9780511804441>
- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research*, 11, 2079-2107.
- Chowdary, B. V., Keerthi, K. L., Sandhya, D., Sunny, Ch. y Reddy, M. D. (2025). Advancing telemarketing success using ensemble-driven online machine learning. *2025 5th International Conference on Soft Computing for Security Applications (ICSCSA)*, 974-978. <https://doi.org/10.1109/icscsa66339.2025.11170856>
- Cliff, N. (1993). Dominance statistics: Ordinal analyses to answer ordinal questions. *Psychological Bulletin*, 114(3), 494-509. <https://doi.org/10.1037/0033-2909.114.3.494>
- DeLong, E. R., DeLong, D. M. y Clarke-Pearson, D. L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves: A nonparametric approach. *Biometrics*, 44(3), 837-845. <https://doi.org/10.2307/2531595>
- Demšar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research*, 7, 1-30.
- Diebold, F. X. y Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253-263. <https://doi.org/10.1080/07350015.1995.10524599>
- Durbin, J. y Koopman, S. J. (2012). *Time series analysis by state space methods* (2.ª ed.). Oxford University Press. <https://doi.org/10.1093/acprof:oso/9780199641178.001.0001>
- Fahrmeir, L. (1992). Posterior mode estimation by extended Kalman filtering for multivariate dynamic generalized linear models. *Journal of the American Statistical Association*, 87(418), 501-509. <https://doi.org/10.1080/01621459.1992.10475232>
- Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M. y Bouchachia, A. (2014). A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), 1-37. <https://doi.org/10.1145/2523813>
- Garg, S., Wu, Y., Balakrishnan, S. y Lipton, Z. C. (2020). A unified view of label shift estimation. *Advances in Neural Information Processing Systems*, 33.
- Grzenda, M., Gomes, H. M. y Bifet, A. (2020). Delayed labelling evaluation for data streams. *Data Mining and Knowledge Discovery*, 34(5), 1237-1266. <https://doi.org/10.1007/s10618-019-00654-y>
- Harries, M. (1999). *SPLICE-2 comparative evaluation: Electricity pricing* [Informe técnico]. University of New South Wales.
- Harvey, D., Leybourne, S. y Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2), 281-291. <https://doi.org/10.1016/S0169-2070(96)00719-4>
- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65-70.
- Kalman, R. E. (1960). A new approach to linear filtering and prediction problems. *Journal of Basic Engineering*, 82(1), 35-45. <https://doi.org/10.1115/1.3662552>
- Lipton, Z. C., Wang, Y.-X. y Smola, A. J. (2018). Detecting and correcting for label shift with black box predictors. *Proceedings of the 35th International Conference on Machine Learning*, PMLR 80, 3122-3130.
- Lundberg, S. M. y Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, 30.
- MacKay, D. J. C. (1992). The evidence framework applied to classification networks. *Neural Computation*, 4(5), 720-736. <https://doi.org/10.1162/neco.1992.4.5.720>
- McCormick, T. H., Raftery, A. E., Madigan, D. y Burd, R. S. (2012). Dynamic logistic regression and dynamic model averaging for binary classification. *Biometrics*, 68(1), 23-30. <https://doi.org/10.1111/j.1541-0420.2011.01645.x>
- Moro, S., Cortez, P. y Rita, P. (2014). A data-driven approach to predict the success of bank telemarketing. *Decision Support Systems*, 62, 22-31. <https://doi.org/10.1016/j.dss.2014.03.001>
- Nadeau, C. y Bengio, Y. (2003). Inference for the generalization error. *Machine Learning*, 52(3), 239-281. <https://doi.org/10.1023/A:1024068626366>
- Politis, D. N. y Romano, J. P. (1994). The stationary bootstrap. *Journal of the American Statistical Association*, 89(428), 1303-1313. <https://doi.org/10.1080/01621459.1994.10476870>
- Romano, J., Kromrey, J. D., Coraggio, J. y Skowronek, J. (2006). Appropriate statistics for ordinal level data: Should we really be using t-test and Cohen's d for evaluating group differences on the NSSE and other surveys? *Annual Meeting of the Florida Association of Institutional Research*.
- Saerens, M., Latinne, P. y Decaestecker, C. (2002). Adjusting the outputs of a classifier to new a priori probabilities: A simple procedure. *Neural Computation*, 14(1), 21-41. <https://doi.org/10.1162/089976602753284446>
- Varma, S. y Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics*, 7, 91. <https://doi.org/10.1186/1471-2105-7-91>
- West, M., Harrison, P. J. y Migon, H. S. (1985). Dynamic generalized linear models and Bayesian forecasting. *Journal of the American Statistical Association*, 80(389), 73-83. <https://doi.org/10.1080/01621459.1985.10477131>
- Zhang, K. y Fan, W. (2008). Forecasting skewed biased stochastic ozone days: Analyses, solutions and beyond. *Knowledge and Information Systems*, 14(3), 299-326. <https://doi.org/10.1007/s10115-007-0095-1>

## Entradas BibTeX citadas

Las citas del libro enlazan a estas entradas de `references.bib`, generadas con `sphinxcontrib-bibtex`.

```{bibliography}
```